<a href="https://colab.research.google.com/github/devpatel0005/Hindi-English-Language-Translation/blob/main/DL50_Encoder_Decoder_Language_Translation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import numpy as np
import tensorflow
from tensorflow import keras
from keras.models import Model
from tensorflow.keras.layers import Input, LSTM,Dense

In [2]:
batch_size=64
epochs=100
latent_dim=256 # Latent dimensionality of the Encoding Space
num_samples=10000  #no.of samples to train on
data_path='/content/hin.txt'

In [3]:
# vectorzie the Data

input_texts=[]
target_texts=[]
input_characters=set()
target_characters=set()

with open(data_path,'r',encoding='utf-8') as f:
  lines=f.read().split('\n') #this will seperate the rows first
for line in lines[:min(num_samples,len(lines)-1)]:
  input_text,target_text,_=line.split('\t')
  # we use tab as the start sequence character and \n as the end sequence character
  target_text='\t'+target_text + '\n'
  input_texts.append(input_text)
  target_texts.append(target_text)
  # This code will seperate the English and the Hindi messages as input and outputtext
  for char in input_text:
    if char not in input_characters:
      input_characters.add(char)
  for char in target_text:
    if char not in target_characters:
      target_characters.add(char)





In [4]:
lines

['Wow!\tवाह!\tCC-BY 2.0 (France) Attribution: tatoeba.org #52027 (Zifre) & #6179147 (fastrizwaan)',
 'Duck!\tझुको!\tCC-BY 2.0 (France) Attribution: tatoeba.org #280158 (CM) & #6179041 (fastrizwaan)',
 'Duck!\tबतख़!\tCC-BY 2.0 (France) Attribution: tatoeba.org #280158 (CM) & #6179042 (fastrizwaan)',
 'Help!\tबचाओ!\tCC-BY 2.0 (France) Attribution: tatoeba.org #435084 (lukaszpp) & #459377 (minshirui)',
 'Jump.\tउछलो.\tCC-BY 2.0 (France) Attribution: tatoeba.org #631038 (Shishir) & #6179121 (fastrizwaan)',
 'Jump.\tकूदो.\tCC-BY 2.0 (France) Attribution: tatoeba.org #631038 (Shishir) & #6179122 (fastrizwaan)',
 'Jump.\tछलांग.\tCC-BY 2.0 (France) Attribution: tatoeba.org #631038 (Shishir) & #6179123 (fastrizwaan)',
 'Hello!\tनमस्ते।\tCC-BY 2.0 (France) Attribution: tatoeba.org #373330 (CK) & #480306 (minshirui)',
 'Hello!\tनमस्कार।\tCC-BY 2.0 (France) Attribution: tatoeba.org #373330 (CK) & #480307 (minshirui)',
 'Cheers!\tवाह-वाह!\tCC-BY 2.0 (France) Attribution: tatoeba.org #487006 (human6

In [5]:
input_texts

['Wow!',
 'Duck!',
 'Duck!',
 'Help!',
 'Jump.',
 'Jump.',
 'Jump.',
 'Hello!',
 'Hello!',
 'Cheers!',
 'Cheers!',
 'Exhale.',
 'Exhale.',
 'Got it?',
 "I'm OK.",
 'Inhale.',
 'Inhale.',
 'Thanks!',
 'We won.',
 'Awesome!',
 'Come in.',
 'Get out!',
 'Go away!',
 'Goodbye!',
 'Perfect!',
 'Perfect!',
 'We lost.',
 'Welcome.',
 'Welcome.',
 'Have fun.',
 'Have fun.',
 'Have fun.',
 'I forgot.',
 'I forgot.',
 "I'll pay.",
 "I'm fine.",
 "I'm full.",
 "Let's go!",
 'Pick Tom.',
 'Answer me.',
 'Birds fly.',
 'Excuse me.',
 'Fantastic!',
 'I fainted.',
 'I fear so.',
 'I laughed.',
 "I'm alone.",
 "I'm alone.",
 "I'm bored.",
 "I'm broke.",
 "I'm tired.",
 "It's cold.",
 'Well done!',
 'Who knows?',
 'Who knows?',
 'Who knows?',
 'Who knows?',
 'Wonderful!',
 'Birds sing.',
 'Come on in.',
 'Definitely!',
 "Don't move.",
 'Fire burns.',
 'Follow him.',
 'I can swim.',
 'I can swim.',
 'I love you.',
 'I love you.',
 'I love you.',
 'I love you.',
 'I love you.',
 "I'm coming.",
 "I'm hung

In [6]:
target_characters

{'\t',
 '\n',
 ' ',
 '!',
 '"',
 '(',
 ')',
 ',',
 '-',
 '.',
 '0',
 '1',
 '7',
 '9',
 ':',
 '?',
 'A',
 'B',
 'I',
 '|',
 'ँ',
 'ं',
 'ः',
 'अ',
 'आ',
 'इ',
 'ई',
 'उ',
 'ऊ',
 'ऋ',
 'ए',
 'ऐ',
 'ऑ',
 'ओ',
 'औ',
 'क',
 'ख',
 'ग',
 'घ',
 'च',
 'छ',
 'ज',
 'झ',
 'ञ',
 'ट',
 'ठ',
 'ड',
 'ढ',
 'ण',
 'त',
 'थ',
 'द',
 'ध',
 'न',
 'प',
 'फ',
 'ब',
 'भ',
 'म',
 'य',
 'र',
 'ल',
 'व',
 'श',
 'ष',
 'स',
 'ह',
 '़',
 'ा',
 'ि',
 'ी',
 'ु',
 'ू',
 'ृ',
 'ॅ',
 'े',
 'ै',
 'ॉ',
 'ो',
 'ौ',
 '्',
 '।',
 '०',
 '१',
 '२',
 '४',
 '५',
 '६',
 '७',
 '८',
 '९',
 '\u200d',
 '€'}

In [7]:
len(target_characters)

93

In [8]:
input_characters=sorted(list(input_characters))
target_characters=sorted(list(target_characters))
num_encoder_tokens=len(input_characters)
num_decoder_tokens=len(target_characters)
max_encoder_seq_length=max([len(txt) for txt in input_texts])
max_decoder_seq_length=max([len(txt) for txt in target_texts])


In [9]:
print('Number of samples:', len(input_texts))
print('Number of unique input tokens:', num_encoder_tokens)
print('Number of unique output tokens:', num_decoder_tokens)
print('Max sequence length for inputs:', max_encoder_seq_length)
print('Max sequence length for outputs:', max_decoder_seq_length)

Number of samples: 3117
Number of unique input tokens: 70
Number of unique output tokens: 93
Max sequence length for inputs: 107
Max sequence length for outputs: 123


In [10]:
input_token_index=dict(
    [(char,i) for i,char in enumerate(input_characters)]
)
target_token_index=dict(
    [(char,i) for i,char in enumerate(target_characters)]
)

In [11]:
target_token_index

{'\t': 0,
 '\n': 1,
 ' ': 2,
 '!': 3,
 '"': 4,
 '(': 5,
 ')': 6,
 ',': 7,
 '-': 8,
 '.': 9,
 '0': 10,
 '1': 11,
 '7': 12,
 '9': 13,
 ':': 14,
 '?': 15,
 'A': 16,
 'B': 17,
 'I': 18,
 '|': 19,
 'ँ': 20,
 'ं': 21,
 'ः': 22,
 'अ': 23,
 'आ': 24,
 'इ': 25,
 'ई': 26,
 'उ': 27,
 'ऊ': 28,
 'ऋ': 29,
 'ए': 30,
 'ऐ': 31,
 'ऑ': 32,
 'ओ': 33,
 'औ': 34,
 'क': 35,
 'ख': 36,
 'ग': 37,
 'घ': 38,
 'च': 39,
 'छ': 40,
 'ज': 41,
 'झ': 42,
 'ञ': 43,
 'ट': 44,
 'ठ': 45,
 'ड': 46,
 'ढ': 47,
 'ण': 48,
 'त': 49,
 'थ': 50,
 'द': 51,
 'ध': 52,
 'न': 53,
 'प': 54,
 'फ': 55,
 'ब': 56,
 'भ': 57,
 'म': 58,
 'य': 59,
 'र': 60,
 'ल': 61,
 'व': 62,
 'श': 63,
 'ष': 64,
 'स': 65,
 'ह': 66,
 '़': 67,
 'ा': 68,
 'ि': 69,
 'ी': 70,
 'ु': 71,
 'ू': 72,
 'ृ': 73,
 'ॅ': 74,
 'े': 75,
 'ै': 76,
 'ॉ': 77,
 'ो': 78,
 'ौ': 79,
 '्': 80,
 '।': 81,
 '०': 82,
 '१': 83,
 '२': 84,
 '४': 85,
 '५': 86,
 '६': 87,
 '७': 88,
 '८': 89,
 '९': 90,
 '\u200d': 91,
 '€': 92}

In [12]:
# We are creating the one_hot_representataion using numpy

encoder_input_data=np.zeros(
    (len(input_texts),max_encoder_seq_length,num_encoder_tokens),
    dtype='float32')

decoder_input_data=np.zeros(
    (len(input_texts),max_decoder_seq_length,num_decoder_tokens),
    dtype='float32')

decoder_target_data=np.zeros(
    (len(input_texts),max_decoder_seq_length,num_decoder_tokens),
    dtype='float32')


In [13]:
for i, (input_text, target_text) in enumerate(zip(input_texts, target_texts)):
    for t, char in enumerate(input_text):
        encoder_input_data[i, t, input_token_index[char]] = 1.
    for t, char in enumerate(target_text):
        # decoder_target_data is ahead of decoder_input_data by one timestep
        # and will not include the start character.
        if t > 0:
            # decoder_target_data will be ahead by one timestep
            decoder_target_data[i, t - 1, target_token_index[char]] = 1.
        decoder_input_data[i, t, target_token_index[char]] = 1.

In [14]:
encoder_input_data[0].shape

(107, 70)

In [15]:
# Define an input sequence and process it.
encoder_inputs = Input(shape=(None, num_encoder_tokens))
encoder = LSTM(latent_dim, return_state=True)
encoder_outputs, state_h, state_c = encoder(encoder_inputs)
# We discard `encoder_outputs` and only keep the states. since in the encode we don't need the state invidual outputs so we keep only state output
encoder_states = [state_h, state_c]


In [16]:

# Set up the decoder, using `encoder_states` as initial state.
decoder_inputs = Input(shape=(None, num_decoder_tokens))
# We set up our decoder to return full output sequences,
# and to return internal states as well. We don't use the
# return states in the training model, but we will use them in inference.
decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(decoder_inputs,
                                  initial_state=encoder_states) # Here in the decoder we need the output for each of the cell so here used the decder_outputs for generating the exact output senquence
decoder_dense = Dense(num_decoder_tokens, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

In [17]:
# Define the model that will turn
# 'encoder_input_data' & 'decoder_input_data' into 'decoder_target_data'
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

# Run training
model.compile(optimizer='rmsprop', loss='categorical_crossentropy',
             metrics=['accuracy'])
model.fit([encoder_input_data, decoder_input_data], decoder_target_data,
          batch_size=batch_size,
          epochs=epochs,
          validation_split=0.2)

Epoch 1/100
39/39 ━━━━━━━━━━━━━━━━━━━━ 6s 44ms/step - accuracy: 0.0432 - loss: 0.9512 - val_accuracy: 0.0758 - val_loss: 1.5035
Epoch 2/100
39/39 ━━━━━━━━━━━━━━━━━━━━ 2s 31ms/step - accuracy: 0.0436 - loss: 0.8710 - val_accuracy: 0.0758 - val_loss: 1.4891
Epoch 3/100
39/39 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.0436 - loss: 0.8642 - val_accuracy: 0.0758 - val_loss: 1.4769
Epoch 4/100
39/39 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.0436 - loss: 0.8624 - val_accuracy: 0.0758 - val_loss: 1.4660
Epoch 5/100
39/39 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - accuracy: 0.0436 - loss: 0.8608 - val_accuracy: 0.0758 - val_loss: 1.4690
Epoch 6/100
39/39 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - accuracy: 0.0436 - loss: 0.8600 - val_accuracy: 0.0758 - val_loss: 1.4599
Epoch 7/100
39/39 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - accuracy: 0.0436 - loss: 0.8588 - val_accuracy: 0.0758 - val_loss: 1.4604
Epoch 8/100
39/39 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.0438 - loss: 0.8591 - val_accuracy: 0.

In [18]:
# Next: inference mode (sampling).
# Here's the drill:
# 1) encode input and retrieve initial decoder state
# 2) run one step of decoder with this initial state
# 3) and a "start of sequence" token as target.
# 4) Output will be the next target token
# 5) Repeat with the current target token and current states

# Define sampling models
encoder_model = Model(encoder_inputs, encoder_states)

decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]
decoder_outputs, state_h, state_c = decoder_lstm(
    decoder_inputs, initial_state=decoder_states_inputs)
decoder_states = [state_h, state_c]
decoder_outputs = decoder_dense(decoder_outputs)
decoder_model = Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs] + decoder_states)


# Reverse-lookup token index to decode sequences back to
# something readable.
reverse_input_char_index = dict(
    (i, char) for char, i in input_token_index.items())
reverse_target_char_index = dict(
    (i, char) for char, i in target_token_index.items())


def decode_sequence(input_seq):
    # Encode the input as state vectors.
    states_value = encoder_model.predict(input_seq)

    # Generate empty target sequence of length 1.
    target_seq = np.zeros((1, 1, num_decoder_tokens))
    # Populate the first character of target sequence with the start character.
    target_seq[0, 0, target_token_index['\t']] = 1.

    # Sampling Loop for a batch of sequences
    # (to simplify, here we assume a batch of size 1).
    stop_condition = False
    decoded_sentence = ''
    while not stop_condition:
        output_tokens, h, c = decoder_model.predict(
            [target_seq] + states_value)

        # Sample a token
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_char = reverse_target_char_index[sampled_token_index]
        decoded_sentence += sampled_char

        # Exit condition: either hit max length
        # or find stop character.
        if (sampled_char == '\n' or
           len(decoded_sentence) > max_decoder_seq_length):
            stop_condition = True
        # Update the target sequence (of length 1).
        target_seq = np.zeros((1, 1, num_decoder_tokens))
        target_seq[0, 0, sampled_token_index] = 1.

        # Update states
        states_value = [h, c]

        return decoded_sentence

for seq_index in range(100):
    # Take one sequence (part of the training set)
    # for trying out decoding.
    input_seq = encoder_input_data[seq_index: seq_index + 1]
    decoded_sentence = decode_sequence(input_seq)
    print('-')
    print('Input sentence:', input_texts[seq_index])
    print('Decoded sentence:', decoded_sentence)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 135ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 157ms/step
-
Input sentence: Wow!
Decoded sentence: म
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
-
Input sentence: Duck!
Decoded sentence: म
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
-
Input sentence: Duck!
Decoded sentence: म
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
-
Input sentence: Help!
Decoded sentence: म
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
-
Input sentence: Jump.
Decoded sentence: म
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
-
Input sentence: Jump.
Decoded sentence: म
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
-
Input sentence: Jump.
Decoded sentence: म
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
-
Input sentence: Hello!
Decoded sentence: म
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
